In [1]:
from neural_net.model_config import get_config, ModelConfig
from neural_net.model_design import train_model
from neural_net.model_data import get_data
from neural_net.utils import set_logger
import tensorflow as tf
from pathlib import Path

2025-07-26 04:59:24.460563: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
workdir = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Run3_0626/Vars_EvenEvs')

roster_name = 'hierarchical'
binary_config_name = 'binary_ttbar_rest_4j'
binary_config = get_config(binary_config_name, roster_name)
logger = set_logger()
_, _, test_data = get_data(binary_config, workdir, logger)
test_data = test_data.prefetch(tf.data.AUTOTUNE)

Available top-level keys: ['SL_4j_resolved_vars', 'models']
Loading models from hierarchical.yml...
	binary_ttbar_rest_4j

Dataset metadata after enriching:
                     File            Tree Process  Total Events  Process Event Ratio  Take events  DS Events  DS Ratio     GenWeight      Class  Class Index  Class DS Ratio   SampleWeight
0         tWminus_dl_2022  SL_4j_resolved      tW         39057             0.008372          167        167  0.002784  6.344046e+02  non_ttbar            0        0.008353     232.248871
1       tWminus_dl_2022EE  SL_4j_resolved      tW        132667             0.028436          568        568  0.009469  2.157735e+03  non_ttbar            0        0.028411     789.924377
2         tWminus_dl_2023  SL_4j_resolved      tW         41630             0.008923          178        178  0.002967  6.761917e+02  non_ttbar            0        0.008904     247.546707
3     tWminus_dl_2023BPix  SL_4j_resolved      tW         20439             0.004381       

In [3]:
from neural_net.model_design import CustomStandardizer, ReplaceUndefinedValuesWithConstant
def load_model(path: Path):
    model = tf.keras.models.load_model(
        path,
        custom_objects={
            "CustomStandardizer": CustomStandardizer,
            "ReplaceUndefinedValuesWithConstant": ReplaceUndefinedValuesWithConstant
        }
    )
    return model

model = load_model(Path("/eos/user/a/anunezde/Z_OUTPUT_eos/Run3_0626/Vars_EvenEvs/NN_hierarchical_test/hierarchicalv2/last_checkpoint"))
print(type(model))

<class 'keras.src.engine.functional.Functional'>


In [4]:
test_dataset = test_data.map(lambda d: {"features": d["features"], "class_oh": d["class_oh"]}).cache()
features = tf.concat([batch["features"] for batch in test_dataset], axis=0).numpy()
true_labels = tf.concat([batch["class_oh"] for batch in test_dataset], axis=0).numpy()

In [5]:
from pathlib import Path
outdir = Path.cwd()

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from neural_net.model_evaluator import *

model_config = binary_config

clss: Classification = get_classification(model, features, true_labels)
mapper = model_config.mapper
feature_names = model_config.features

metrics = get_metrics(clss, mapper)
log_msg("\n=== Model Evaluation Summary ===")
for key, value in metrics.items():
    if isinstance(value, float):
        log_msg(f"{key}: {value:.4f}")

plot_confusion_matrix(metrics['confusion_matrix_norm_true'], mapper, outdir, "Confusion Matrix (Normalized by True)", "confusion_matrix_true")
plot_confusion_matrix(metrics['confusion_matrix_norm_pred'], mapper, outdir, "Confusion Matrix (Normalized by Pred)", "confusion_matrix_pred")
plot_roc_curves(clss.true_labels, clss.probabilities, mapper, outdir)
plot_score_distributions(clss.true_labels, clss.probabilities, mapper, outdir)
plot_correlation_matrix(features, feature_names, outdir)


375/375 [==============================] - 0s 1ms/step
Youden optimal threshold: 0.6663 (J=0.1071)

=== Model Evaluation Summary ===
accuracy: 0.5400
balanced_accuracy: 0.5535
loss: 0.6913
precision_macro: 0.5477
recall_macro: 0.5535
f1_macro: 0.5303
precision_micro: 0.5400
recall_micro: 0.5400
f1_micro: 0.5400
precision_weighted: 0.6040
recall_weighted: 0.5400
f1_weighted: 0.5529
mcc: 0.1010
kappa: 0.0937
auc_roc: 0.5544
auc_pr: 0.7247
brier_score: 0.2460
ece: 0.1268
optimal_threshold: 0.6663
accuracy_at_threshold: 0.5400
sensitivity_at_threshold: 0.5131
specificity_at_threshold: 0.5939
precision_at_threshold: 0.7165
f1_at_threshold: 0.5980
youden_j: 0.1070
Confusion Matrix (Normalized by True) saved to /afs/cern.ch/user/a/anunezde/bamboodev/hh/confusion_matrix_true.pdf.pdf
Confusion Matrix (Normalized by Pred) saved to /afs/cern.ch/user/a/anunezde/bamboodev/hh/confusion_matrix_pred.pdf.pdf
ROC curves saved to /afs/cern.ch/user/a/anunezde/bamboodev/hh/roc_curves.pdf
Score distribution